In [ ]:
"""
Gradient Boosting Classifier — From Scratch
============================================
Implementasi lengkap Gradient Boosting untuk klasifikasi biner
menggunakan Decision Tree Regressor sebagai weak learner.

Referensi:
  Friedman, J.H. (2001). Greedy Function Approximation: A Gradient Boosting Machine.
  Annals of Statistics, 29(5), 1189–1232.
"""

import numpy as np
from collections import Counter


# ─────────────────────────────────────────────────────────────
# 1. Decision Tree Node & Regressor (weak learner)
# ─────────────────────────────────────────────────────────────

class TreeNode:
    """Satu simpul di dalam decision tree."""

    def __init__(self):
        self.feature_idx = None   # indeks fitur yang dipakai untuk split
        self.threshold   = None   # nilai ambang batas split
        self.left        = None   # anak kiri (nilai <= threshold)
        self.right       = None   # anak kanan (nilai > threshold)
        self.value       = None   # nilai prediksi (hanya untuk leaf)

    @property
    def is_leaf(self):
        return self.value is not None


class DecisionTreeRegressor:
    """
    Decision Tree Regressor minimalis.
    Dipakai sebagai 'weak learner' di dalam Gradient Boosting.

    Parameters
    ----------
    max_depth : int
        Kedalaman maksimum pohon.
    min_samples_split : int
        Jumlah sampel minimum agar sebuah simpul di-split.
    """

    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    # ── Fitting ──────────────────────────────────────────────

    def fit(self, X, y):
        self.root = self._build(X, y, depth=0)
        return self

    def _build(self, X, y, depth):
        node = TreeNode()

        # Kondisi berhenti
        if (depth >= self.max_depth or
                len(y) < self.min_samples_split or
                np.var(y) == 0):
            node.value = np.mean(y)
            return node

        # Cari split terbaik
        best = self._best_split(X, y)
        if best is None:
            node.value = np.mean(y)
            return node

        feat, thr, left_idx, right_idx = best
        node.feature_idx = feat
        node.threshold   = thr
        node.left  = self._build(X[left_idx], y[left_idx],  depth + 1)
        node.right = self._build(X[right_idx], y[right_idx], depth + 1)
        return node

    def _best_split(self, X, y):
        """Cari feature + threshold terbaik berdasarkan MSE terkecil."""
        best_mse  = float('inf')
        best_feat = None
        best_thr  = None
        best_left = None
        best_right = None

        n_samples, n_features = X.shape
        for feat in range(n_features):
            col = X[:, feat]
            thresholds = np.unique(col)
            for thr in thresholds:
                left_idx  = np.where(col <= thr)[0]
                right_idx = np.where(col >  thr)[0]
                if len(left_idx) == 0 or len(right_idx) == 0:
                    continue
                mse = self._weighted_mse(y, left_idx, right_idx)
                if mse < best_mse:
                    best_mse   = mse
                    best_feat  = feat
                    best_thr   = thr
                    best_left  = left_idx
                    best_right = right_idx

        if best_feat is None:
            return None
        return best_feat, best_thr, best_left, best_right

    @staticmethod
    def _weighted_mse(y, left_idx, right_idx):
        """MSE total berbobot ukuran partisi."""
        n = len(y)
        mse_l = np.var(y[left_idx])  * len(left_idx)  / n
        mse_r = np.var(y[right_idx]) * len(right_idx) / n
        return mse_l + mse_r

    # ── Prediksi ─────────────────────────────────────────────

    def predict(self, X):
        return np.array([self._traverse(x, self.root) for x in X])

    def _traverse(self, x, node):
        if node.is_leaf:
            return node.value
        if x[node.feature_idx] <= node.threshold:
            return self._traverse(x, node.left)
        return self._traverse(x, node.right)


# ─────────────────────────────────────────────────────────────
# 2. Gradient Boosting Classifier
# ─────────────────────────────────────────────────────────────

class GradientBoostingClassifier:
    """
    Gradient Boosting Classifier for binary classification.

    Menggunakan:
      - Loss     : Binary Cross-Entropy (log-loss)
      - Gradien  : Residual = y - p  (negatif gradien log-loss)
      - Update   : F_m(x) = F_{m-1}(x) + learning_rate * h_m(x)
      - Output   : sigmoid(F(x)) → probabilitas kelas 1

    Parameters
    ----------
    n_estimators  : int   — jumlah pohon (iterasi boosting)
    learning_rate : float — shrinkage factor (η)
    max_depth     : int   — kedalaman tiap pohon
    min_samples_split : int
    """

    def __init__(
        self,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        min_samples_split=2,
    ):
        self.n_estimators      = n_estimators
        self.learning_rate     = learning_rate
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split

        self.trees_     = []      # daftar weak learner
        self.F0_        = None    # prediksi awal (log-odds)
        self.train_loss_= []      # log-loss per iterasi (untuk monitoring)

    # ── Fungsi bantu ────────────────────────────────────────

    @staticmethod
    def _sigmoid(z):
        """Sigmoid yang stabil secara numerik."""
        return np.where(
            z >= 0,
            1 / (1 + np.exp(-z)),
            np.exp(z) / (1 + np.exp(z))
        )

    @staticmethod
    def _log_loss(y, p):
        """Binary cross-entropy."""
        eps = 1e-15
        p   = np.clip(p, eps, 1 - eps)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    @staticmethod
    def _negative_gradient(y, p):
        """
        Negatif gradien dari log-loss terhadap F(x).
        r_i = y_i - p_i  (residual / pseudo-residual)
        """
        return y - p

    # ── Fitting ──────────────────────────────────────────────

    def fit(self, X, y):
        """
        Latih model Gradient Boosting.

        Parameters
        ----------
        X : ndarray, shape (n_samples, n_features)
        y : ndarray, shape (n_samples,), nilai 0 atau 1
        """
        y = np.asarray(y, dtype=float)
        n = len(y)

        # ── Langkah 0: Prediksi awal F0 = log(p̄/(1-p̄))
        p_bar   = np.mean(y)
        p_bar   = np.clip(p_bar, 1e-15, 1 - 1e-15)
        self.F0_= np.log(p_bar / (1 - p_bar))
        F       = np.full(n, self.F0_)          # F_0(x) untuk semua sampel

        self.trees_ = []
        self.train_loss_ = []

        for m in range(self.n_estimators):
            # ── Langkah 1: Hitung probabilitas saat ini
            p = self._sigmoid(F)

            # ── Catat log-loss
            self.train_loss_.append(self._log_loss(y, p))

            # ── Langkah 2: Hitung pseudo-residual (negatif gradien)
            r = self._negative_gradient(y, p)

            # ── Langkah 3: Fit satu Decision Tree Regressor ke residual
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
            )
            tree.fit(X, r)

            # ── Langkah 4: Update prediksi F
            h = tree.predict(X)
            F = F + self.learning_rate * h
            self.trees_.append(tree)

        return self

    # ── Prediksi ─────────────────────────────────────────────

    def predict_proba(self, X):
        """Kembalikan probabilitas [P(y=0), P(y=1)] untuk setiap sampel."""
        X   = np.asarray(X)
        F   = np.full(len(X), self.F0_)
        for tree in self.trees_:
            F += self.learning_rate * tree.predict(X)
        p1  = self._sigmoid(F)
        return np.column_stack([1 - p1, p1])

    def predict(self, X, threshold=0.5):
        """Prediksi label kelas (0 atau 1) menggunakan threshold."""
        proba = self.predict_proba(X)[:, 1]
        return (proba >= threshold).astype(int)

    # ── Evaluasi ─────────────────────────────────────────────

    def score(self, X, y):
        """Akurasi prediksi."""
        return np.mean(self.predict(X) == np.asarray(y))


# ─────────────────────────────────────────────────────────────
# 3. Fungsi Evaluasi Tambahan
# ─────────────────────────────────────────────────────────────

def confusion_matrix(y_true, y_pred):
    """Confusion matrix 2×2: [[TN, FP], [FN, TP]]."""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    TP = np.sum((y_pred == 1) & (y_true == 1))
    TN = np.sum((y_pred == 0) & (y_true == 0))
    FP = np.sum((y_pred == 1) & (y_true == 0))
    FN = np.sum((y_pred == 0) & (y_true == 1))
    return np.array([[TN, FP], [FN, TP]])


def classification_report(y_true, y_pred):
    """Laporan presisi, recall, F1, dan akurasi."""
    cm           = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    precision    = TP / (TP + FP + 1e-15)
    recall       = TP / (TP + FN + 1e-15)
    f1           = 2 * precision * recall / (precision + recall + 1e-15)
    accuracy     = (TP + TN) / len(y_true)
    return {
        "accuracy" : accuracy,
        "precision": precision,
        "recall"   : recall,
        "f1_score" : f1,
        "confusion_matrix": cm,
    }


def train_test_split(X, y, test_size=0.2, random_state=42):
    """Bagi data menjadi train dan test."""
    rng  = np.random.default_rng(random_state)
    idx  = rng.permutation(len(y))
    cut  = int(len(y) * (1 - test_size))
    tr, te = idx[:cut], idx[cut:]
    return X[tr], X[te], y[tr], y[te]


# ─────────────────────────────────────────────────────────────
# 4. Demo — Breast Cancer dataset sederhana (XOR-like)
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    np.random.seed(42)

    # ── Buat dataset sintetis (two-moon style)
    from sklearn.datasets import make_classification, make_moons

    print("=" * 60)
    print("  Gradient Boosting Classifier — From Scratch")
    print("=" * 60)

    # Dataset 1: linear separable
    print("\n[1] Dataset: make_classification")
    X, y = make_classification(
        n_samples=500, n_features=10, n_informative=5,
        n_redundant=2, random_state=42
    )
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    clf = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
    )
    clf.fit(X_train, y_train)
    report = classification_report(y_test, clf.predict(X_test))

    print(f"  Accuracy : {report['accuracy']:.4f}")
    print(f"  Precision: {report['precision']:.4f}")
    print(f"  Recall   : {report['recall']:.4f}")
    print(f"  F1-Score : {report['f1_score']:.4f}")
    print(f"  Confusion Matrix:\n{report['confusion_matrix']}")
    print(f"  Final train loss: {clf.train_loss_[-1]:.4f}")

    # Dataset 2: non-linear (moons)
    print("\n[2] Dataset: make_moons (non-linear)")
    X2, y2 = make_moons(n_samples=500, noise=0.2, random_state=42)
    X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2)

    clf2 = GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
    )
    clf2.fit(X2_train, y2_train)
    report2 = classification_report(y2_test, clf2.predict(X2_test))

    print(f"  Accuracy : {report2['accuracy']:.4f}")
    print(f"  Precision: {report2['precision']:.4f}")
    print(f"  Recall   : {report2['recall']:.4f}")
    print(f"  F1-Score : {report2['f1_score']:.4f}")
    print(f"  Final train loss: {clf2.train_loss_[-1]:.4f}")

    print("\nSelesai.")

  Gradient Boosting Classifier — From Scratch

[1] Dataset: make_classification
  Accuracy : 0.8700
  Precision: 0.9375
  Recall   : 0.8182
  F1-Score : 0.8738
  Confusion Matrix:
[[42  3]
 [10 45]]
  Final train loss: 0.2271

[2] Dataset: make_moons (non-linear)
  Accuracy : 0.9500
  Precision: 0.9375
  Recall   : 0.9574
  F1-Score : 0.9474
  Final train loss: 0.1821

Selesai.
